# TEND Dataset Explorer

Interactive notebook to inspect generated TEND CSV datasets under `data/TEND/`.

**What you'll find here:**
- Auto-discovery of timestamped `spider_<split>_*.csv` files and companion `.summary.json` files
- Row/column overview, missing values, and boolean quality rates
- Breakdowns by `db_id`, query complexity (joins/aggregations), and conversion warnings
- Side-by-side SQL vs Mongo examples
- A small playground to filter and inspect individual rows

Run cells top-to-bottom. Adjust `SELECTED_SPLIT` or `SAMPLE_DB` in later cells to focus on specific data.

In [1]:
from __future__ import annotations

import json
import re
import textwrap
from pathlib import Path

import pandas as pd

# Ensure project root is on path when launched from TEND/
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "TEND").exists() and (PROJECT_ROOT.parent / "TEND").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

TEND_DATA_DIR = PROJECT_ROOT / "data" / "TEND"
print(f"Project root: {PROJECT_ROOT}")
print(f"TEND data dir: {TEND_DATA_DIR}")

Project root: /Volumes/Work/CodeGen-Implementations-May_26
TEND data dir: /Volumes/Work/CodeGen-Implementations-May_26/data/TEND


In [2]:
def discover_tend_datasets(data_dir: Path) -> pd.DataFrame:
    """List CSV + summary JSON pairs under data/TEND."""
    rows = []
    for csv_path in sorted(data_dir.glob("*.csv")):
        summary_path = csv_path.with_suffix(".summary.json")
        summary = {}
        if summary_path.exists():
            summary = json.loads(summary_path.read_text(encoding="utf-8"))

        # spider_train_0614_1752.csv -> dataset=spider, split=train
        stem = csv_path.stem
        parts = stem.split("_")
        dataset = parts[0] if parts else "unknown"
        split = parts[1] if len(parts) > 1 else "unknown"

        rows.append(
            {
                "dataset": dataset,
                "split": split,
                "csv_path": str(csv_path),
                "summary_path": str(summary_path) if summary_path.exists() else None,
                "size_mb": round(csv_path.stat().st_size / (1024 * 1024), 2),
                **summary,
            }
        )
    return pd.DataFrame(rows)


catalog = discover_tend_datasets(TEND_DATA_DIR)
if catalog.empty:
    raise FileNotFoundError(
        f"No TEND CSV files found in {TEND_DATA_DIR}. "
        "Generate one with: python -m TEND.run_tend --dataset spider --split train --no-eval"
    )

display(catalog)

,dataset,split,csv_path,summary_path,size_mb,total_rows,conversion_success_rate,schema_correct_rate,query_correct_rate,overall_correct_rate
0,spider,train,/Volumes/Work/CodeGen-Implementations-May_26/d...,/Volumes/Work/CodeGen-Implementations-May_26/d...,22.85,7000,0.999714,0.976286,0.976286,0.976286
1,spider,validation,/Volumes/Work/CodeGen-Implementations-May_26/d...,/Volumes/Work/CodeGen-Implementations-May_26/d...,2.87,1034,1.000000,0.980658,0.980658,0.980658


In [3]:
# Pick which split to explore (latest file for that split wins)
SELECTED_SPLIT = "train"  # try: "train", "validation", "dev", "test"

split_files = catalog[catalog["split"] == SELECTED_SPLIT].copy()
if split_files.empty:
    raise ValueError(f"No files for split '{SELECTED_SPLIT}'. Available: {sorted(catalog['split'].unique())}")

selected = split_files.sort_values("csv_path").iloc[-1]
CSV_PATH = Path(selected["csv_path"])
print(f"Loading: {CSV_PATH.name} ({selected['size_mb']} MB)")

df = pd.read_csv(CSV_PATH)
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
df.head(3)

Loading: spider_train_0614_1752.csv (22.85 MB)
Rows: 7,000 | Columns: 15


,source,db_id,question,sql_schema,sql_query,nosql_schema,nosql_query,metadata,conversion_success,schema_correct,query_correct,overall_correct,schema_reason,query_reason,evaluation_response
0,spider,department_management,How many heads of the departments are older th...,CREATE TABLE department (\n Department_ID R...,SELECT count(*) FROM head WHERE age > 56,"{\n ""department"": {\n ""Budget_in_Billions""...","db.head.aggregate(\n[\n {\n ""$match"": {\n ...","{""aggregations"": 1, ""conversion_success"": true...",True,True,True,True,The MongoDB schema is more concise and easier ...,The MongoDB query uses `$match` for filtering ...,"{\n ""schema_correct"": true,\n ""query_correct..."
1,spider,department_management,"List the name, born state and age of the heads...",CREATE TABLE department (\n Department_ID R...,"SELECT name , born_state , age FROM head ORD...","{\n ""department"": {\n ""Budget_in_Billions""...","db.head.find(\n {},\n { name: 1, born_state:...","{""aggregations"": 0, ""conversion_success"": true...",True,True,True,True,The provided SQL and MongoDB schemas are seman...,Both queries retrieve data from the 'head' col...,"{\n ""schema_correct"": true,\n ""query_correct..."
2,spider,department_management,"List the creation year, name and budget of eac...",CREATE TABLE department (\n Department_ID R...,"SELECT creation , name , budget_in_billions ...","{\n ""department"": {\n ""Budget_in_Billions""...","db.department.find(\n {},\n { creation: 1, n...","{""aggregations"": 0, ""conversion_success"": true...",True,True,True,True,The MongoDB schema is more concise and easier ...,The MongoDB query uses a single `find` operati...,"```json\n{\n ""schema_correct"": true,\n ""quer..."


## Dataset overview

In [4]:
overview = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null": df.notna().sum().values,
        "null_pct": (df.isna().mean() * 100).round(2).values,
        "unique": [df[c].nunique(dropna=True) for c in df.columns],
    }
)
display(overview)

print("\nText length stats (characters):")
for col in ["question", "sql_schema", "sql_query", "nosql_schema", "nosql_query"]:
    if col in df.columns:
        lengths = df[col].fillna("").astype(str).str.len()
        print(
            f"  {col:14s} min={lengths.min():5d}  median={lengths.median():6.0f}  "
            f"max={lengths.max():6d}  mean={lengths.mean():6.0f}"
        )

,column,dtype,non_null,null_pct,unique
0,source,object,7000,0.00,1
1,db_id,object,7000,0.00,140
2,question,object,7000,0.00,6962
3,sql_schema,object,7000,0.00,140
4,sql_query,object,7000,0.00,3964
5,nosql_schema,object,7000,0.00,140
6,nosql_query,object,6998,0.03,3527
7,metadata,object,7000,0.00,7000
8,conversion_success,bool,7000,0.00,2
9,schema_correct,bool,7000,0.00,2



Text length stats (characters):
  question       min=   16  median=    69  max=   224  mean=    71
  sql_schema     min=  188  median=   589  max=  6472  mean=   906
  sql_query      min=   18  median=    93  max=   577  mean=   110
  nosql_schema   min=  267  median=   759  max=  9356  mean=  1142
  nosql_query    min=    0  median=    82  max=   364  mean=    91


## Quality metrics

In [ ]:
BOOL_COLS = [c for c in ["conversion_success", "schema_correct", "query_correct", "overall_correct"] if c in df.columns]

def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin({"true", "1", "yes"})

quality = {}
for col in BOOL_COLS:
    quality[col] = to_bool(df[col]).mean()

quality_df = pd.DataFrame({"metric": list(quality.keys()), "rate": list(quality.values())})
quality_df["rate_pct"] = (quality_df["rate"] * 100).round(2)
display(quality_df)

if selected.get("summary_path"):
    print("\nCompanion summary.json:")
    print(json.dumps(json.loads(Path(selected["summary_path"]).read_text()), indent=2))

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    db_counts = df["db_id"].value_counts().head(15)
    db_counts.plot(kind="barh", ax=axes[0], color="#4C78A8")
    axes[0].set_title("Top 15 databases by row count")
    axes[0].set_xlabel("rows")

    if BOOL_COLS:
        quality_df.set_index("metric")["rate"].plot(kind="bar", ax=axes[1], color="#F58518")
        axes[1].set_ylim(0, 1.05)
        axes[1].set_title("Quality rates")
        axes[1].tick_params(axis="x", rotation=30)
    else:
        axes[1].text(0.5, 0.5, "No eval columns (--no-eval run?)", ha="center", va="center")
        axes[1].set_axis_off()

    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — skipping charts. pip install matplotlib")

## Metadata exploration

The `metadata` column is a JSON blob with structural validation flags, join/aggregation counts, and conversion warnings.

In [ ]:
def parse_metadata(value) -> dict:
    if pd.isna(value) or value == "":
        return {}
    try:
        return json.loads(value)
    except json.JSONDecodeError:
        return {"_parse_error": True, "raw": str(value)[:200]}


meta_df = pd.json_normalize(df["metadata"].map(parse_metadata))
meta_df.index = df.index

print("Metadata fields:", list(meta_df.columns))
display(meta_df.head(3))

numeric_cols = [c for c in ["joins", "aggregations", "from_clauses", "tables"] if c in meta_df.columns]
if numeric_cols:
    print("\nQuery complexity summary:")
    display(meta_df[numeric_cols].describe().round(2))

bool_meta_cols = [c for c in meta_df.columns if c.endswith("_valid") or c == "conversion_success"]
if bool_meta_cols:
    print("\nStructural validation pass rates (from metadata):")
    for col in bool_meta_cols:
        rate = to_bool(meta_df[col]).mean()
        print(f"  {col:22s} {rate*100:5.1f}%")

In [ ]:
def warning_stats(meta_series: pd.Series) -> pd.DataFrame:
    counts: dict[str, int] = {}
    rows_with_warnings = 0
    for raw in meta_series:
        meta = parse_metadata(raw)
        warnings = meta.get("conversion_warnings", [])
        if warnings:
            rows_with_warnings += 1
        for w in warnings:
            key = str(w).strip()[:120]
            counts[key] = counts.get(key, 0) + 1
    out = pd.DataFrame(
        [{"warning": k, "count": v} for k, v in sorted(counts.items(), key=lambda x: -x[1])]
    )
    print(f"Rows with conversion warnings: {rows_with_warnings:,} / {len(meta_series):,}")
    return out.head(20)


display(warning_stats(df["metadata"]))

## Sample inspection

Pick a row and compare SQL vs Mongo outputs side by side.

In [ ]:
def show_sample(row: pd.Series, wrap: int = 100) -> None:
    def block(title: str, text: str) -> None:
        print(f"\n{'=' * 80}\n{title}\n{'=' * 80}")
        print(textwrap.fill(str(text), width=wrap) if len(str(text)) > wrap * 3 else text)

    block("Question", row.get("question", ""))
    block(f"db_id: {row.get('db_id', '')} | source: {row.get('source', '')}", "")
    block("SQL schema", row.get("sql_schema", ""))
    block("SQL query", row.get("sql_query", ""))
    block("NoSQL schema", row.get("nosql_schema", ""))
    block("NoSQL query", row.get("nosql_query", ""))

    meta = parse_metadata(row.get("metadata", ""))
    block("Metadata", json.dumps(meta, indent=2))

    for col in BOOL_COLS:
        print(f"{col}: {row.get(col)}")
    for col in ["schema_reason", "query_reason"]:
        if col in row and pd.notna(row[col]):
            block(col, row[col])


SAMPLE_INDEX = 0  # change to any row index
show_sample(df.iloc[SAMPLE_INDEX])

## Issue hunting

Surface rows that failed conversion, structural validation, or Qwen evaluation.

In [ ]:
work = df.copy()
work["meta"] = work["metadata"].map(parse_metadata)

if "conversion_success" in work.columns:
    work["conversion_ok"] = to_bool(work["conversion_success"])
else:
    work["conversion_ok"] = work["meta"].map(lambda m: bool(m.get("conversion_success")))

work["has_warnings"] = work["meta"].map(lambda m: bool(m.get("conversion_warnings")))
work["joins"] = work["meta"].map(lambda m: m.get("joins", 0))
work["aggregations"] = work["meta"].map(lambda m: m.get("aggregations", 0))

issue_summary = pd.DataFrame(
    {
        "issue": [
            "conversion failed",
            "has conversion warnings",
            "contains JOIN (sql)",
            "has aggregations",
            "empty nosql_query",
            "empty nosql_schema",
        ],
        "count": [
            (~work["conversion_ok"]).sum(),
            work["has_warnings"].sum(),
            (work["joins"] > 0).sum(),
            (work["aggregations"] > 0).sum(),
            work["nosql_query"].fillna("").astype(str).str.strip().eq("").sum(),
            work["nosql_schema"].fillna("").astype(str).str.strip().eq("").sum(),
        ],
    }
)
issue_summary["pct"] = (issue_summary["count"] / len(work) * 100).round(2)
display(issue_summary)

if "overall_correct" in work.columns:
    eval_fail = work[~to_bool(work["overall_correct"])]
    print(f"\nQwen overall_correct=False: {len(eval_fail):,} rows")
    display(
        eval_fail[["db_id", "question", "conversion_success", "schema_correct", "query_correct"]]
        .head(10)
    )

conv_fail = work[~work["conversion_ok"]]
print(f"\nConversion failures: {len(conv_fail):,} rows")
if not conv_fail.empty:
    display(conv_fail[["db_id", "question", "sql_query", "nosql_query"]].head(10))

## Playground — filter and inspect

Tweak the filters below to explore subsets of the data.

In [ ]:
SAMPLE_DB = None          # e.g. "concert_singer"
MIN_JOINS = None          # e.g. 1
ONLY_FAILURES = False     # True to show conversion failures only
SEARCH_TEXT = None        # substring match in question or sql_query
MAX_ROWS = 5

filtered = work.copy()
if SAMPLE_DB:
    filtered = filtered[filtered["db_id"] == SAMPLE_DB]
if MIN_JOINS is not None:
    filtered = filtered[filtered["joins"] >= MIN_JOINS]
if ONLY_FAILURES:
    filtered = filtered[~filtered["conversion_ok"]]
if SEARCH_TEXT:
    pattern = re.compile(re.escape(SEARCH_TEXT), re.IGNORECASE)
    mask = filtered["question"].fillna("").str.contains(pattern) | filtered["sql_query"].fillna("").str.contains(pattern)
    filtered = filtered[mask]

print(f"Matched {len(filtered):,} rows")
cols = ["db_id", "question", "conversion_success", "joins", "aggregations"]
if "overall_correct" in filtered.columns:
    cols.append("overall_correct")
display(filtered[cols].head(MAX_ROWS))

if not filtered.empty:
    print("\n--- First matched row detail ---")
    show_sample(filtered.iloc[0])

## Compare train vs validation (if both exist)

In [ ]:
def load_latest_for_split(split: str) -> pd.DataFrame | None:
    subset = catalog[catalog["split"] == split]
    if subset.empty:
        return None
    path = Path(subset.sort_values("csv_path").iloc[-1]["csv_path"])
    return pd.read_csv(path)


splits_to_compare = [s for s in ["train", "validation"] if s in set(catalog["split"])]
compare_rows = []
for split in splits_to_compare:
    part = load_latest_for_split(split)
    if part is None:
        continue
    row = {"split": split, "rows": len(part), "databases": part["db_id"].nunique()}
    for col in BOOL_COLS:
        if col in part.columns:
            row[col] = round(to_bool(part[col]).mean(), 4)
    compare_rows.append(row)

if compare_rows:
    display(pd.DataFrame(compare_rows))
else:
    print("Need at least one split file to compare.")